In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, confusion_matrix
from scipy.spatial.distance import cdist

class AIS_ClonalSelection:
    def __init__(self, num_detectors=200, generations=20, mutation_rate=0.05, random_seed=42):
        self.num_detectors = num_detectors
        self.generations = generations
        self.mutation_rate = mutation_rate
        self.detectors = [] 
        self.scaler = MinMaxScaler()
        self.pca = PCA(n_components=20) 
        self.rng = np.random.default_rng(random_seed)

    def preprocess(self, X, train=True):
        if train:
            X_pca = self.pca.fit_transform(X)
            return self.scaler.fit_transform(X_pca)
        else:
            X_pca = self.pca.transform(X)
            return self.scaler.transform(X_pca)

    def calculate_fitness(self, detectors, self_data, disease_data):
        """
        Fitness Logic:
        1. Reward: Closeness to Disease (Minimize distance to nearest Disease sample)
        2. Penalty: Closeness to Normal (Must be > safety_margin from nearest Normal)
        """
        fitness_scores = []
        
        # Distances to all Normal samples
        dists_self = cdist(detectors, self_data, metric='euclidean')
        min_dist_self = dists_self.min(axis=1) # The closest Normal sample to this detector
        
        # Distances to all Disease samples
        dists_disease = cdist(detectors, disease_data, metric='euclidean')
        min_dist_disease = dists_disease.min(axis=1) # The closest Disease sample
        
        for i in range(len(detectors)):
            # Safety Margin: If detector is too close to a Normal sample it gets removed (fitness = 0)
            if min_dist_self[i] < 0.6: 
                fitness_scores.append(0)
            else:
                # Score, 1e-6 added to avoid division by zero
                score = 1.0 / (min_dist_disease[i] + 1e-6)
                fitness_scores.append(score)
                
        return np.array(fitness_scores)

    def fit(self, normal_data, disease_data):
        print(f"Clonal Selection: Evolving {self.num_detectors} Detectors")
        feature_dim = normal_data.shape[1]
        
        # 1. Initialize Random Population
        random_indices = self.rng.integers(0, len(disease_data), size=self.num_detectors)
        population = disease_data[random_indices] + self.rng.normal(0, 0.1, (self.num_detectors, feature_dim))
        population = np.clip(population, 0, 1) # Keep within bounds
        
        for gen in range(self.generations):
            # 2. Evaluate Fitness
            fitness = self.calculate_fitness(population, normal_data, disease_data)
            
            # Sort by fitness (descending)
            sorted_indices = np.argsort(fitness)[::-1]
            population = population[sorted_indices]
            fitness = fitness[sorted_indices]
            
            if gen % 50 == 0:
                print(f"   Gen {gen}: Best Fitness = {fitness[0]:.4f} (Avg: {np.mean(fitness):.4f})")
            
            # 3. Selection (Keep top 50%)
            survivor_count = int(self.num_detectors * 0.5)
            survivors = population[:survivor_count]
            
            # 4. Cloning & Mutation
            clones_needed = self.num_detectors - survivor_count
            
            # Clone from the best survivors
            parent_indices = self.rng.integers(0, survivor_count, size=clones_needed)
            clones = survivors[parent_indices].copy()
            
            # Mutate: Add random noise to clones
            current_mutation = self.mutation_rate * (1 - (gen / self.generations))
            noise = self.rng.normal(0, current_mutation, clones.shape)
            clones = clones + noise
            clones = np.clip(clones, 0, 1)
            
            population = np.vstack((survivors, clones))
            
        self.detectors = population
        print("Evolution Complete.")

    def predict(self, samples):
        # A sample is "Disease" if it is close to any of our evolved detectors
        detection_radius = 0.7
        
        dists = cdist(samples, self.detectors, metric='euclidean')
        min_dists = dists.min(axis=1)
        
        # If the sample is within radius of a detector, it's flagged as Disease (1)
        predictions = (min_dists < detection_radius).astype(int)
        return predictions



In [ ]:
df = pd.read_csv('GSE33000_Top10000_Var.csv', index_col=0)
labels = df['Diagnosis']
data = df.drop('Diagnosis', axis=1).values

# Indices
normal_indices = np.where(labels.str.contains("C"))[0]
disease_indices = np.where(~labels.str.contains("C"))[0]

# Split Data
n_split = int(len(normal_indices) * 0.7)
d_split = int(len(disease_indices) * 0.7)

train_normal_idx = normal_indices[:n_split]
test_normal_idx = normal_indices[n_split:]

train_disease_idx = disease_indices[:d_split]
test_disease_idx = disease_indices[d_split:]

X_train_normal = data[train_normal_idx]
X_train_disease = data[train_disease_idx]

X_test = data[np.concatenate([test_normal_idx, test_disease_idx])]
y_test = np.array([0]*len(test_normal_idx) + [1]*len(test_disease_idx))

print(f"Training: {len(X_train_normal)} Normal, {len(X_train_disease)} Disease")
print(f"Testing:  {len(test_normal_idx)} Normal, {len(test_disease_idx)} Disease")

# Initialize and Run
ais = AIS_ClonalSelection(num_detectors=1000, generations=100, mutation_rate=0.4)

X_combined_train = np.vstack((X_train_normal, X_train_disease))
ais.preprocess(X_combined_train, train=True)

X_norm_scaled = ais.preprocess(X_train_normal, train=False)
X_dis_scaled = ais.preprocess(X_train_disease, train=False)

# Evolve Detectors
ais.fit(X_norm_scaled, X_dis_scaled)

# Predict
X_test_scaled = ais.preprocess(X_test, train=False)
y_pred = ais.predict(X_test_scaled)

# Results
acc = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print(f"\nAccuracy: {acc:.2%}")
print("\nConfusion Matrix:")
print(f"True Normal:      {cm[0][0]}")
print(f"False Positive:   {cm[0][1]}")
print(f"False Negative:   {cm[1][0]}")
print(f"True Positive:    {cm[1][1]}")